# Day 18 · Colab 1 — Claude Agent + FastAPI Backend + Redis Memory

Merged notebook with **Extension Task 2 (TTL & Forget Tool)**.

## Step 1 — Install dependencies & API key

In [ ]:
!pip install -q anthropic fastapi 'httpx<0.28' fakeredis 2>/dev/null
import os
# os.environ["ANTHROPIC_API_KEY"]="YOUR_KEY"
LIVE=bool(os.environ.get("ANTHROPIC_API_KEY"))
MODEL="claude-sonnet-4-6"
print("LIVE" if LIVE else "OFFLINE")

## Step 2 — FastAPI backend

In [ ]:
from fastapi import FastAPI,HTTPException
from fastapi.testclient import TestClient
app=FastAPI()
_ORDERS={
'A1001':{'id':'A1001','item':'Mechanical keyboard','qty':1,'status':'shipped','total':129.0},
'A1002':{'id':'A1002','item':'USB-C hub','qty':2,'status':'processing','total':58.0},
'A1003':{'id':'A1003','item':'4K monitor','qty':1,'status':'delivered','total':410.0},
}
@app.get('/orders/{order_id}')
def get_order(order_id:str):
    o=_ORDERS.get(order_id.upper())
    if not o:
        raise HTTPException(status_code=404,detail='order not found')
    return o
client=TestClient(app)
client.get('/orders/A1001').json()

## Step 3 — Redis memory

In [ ]:
import json,fakeredis,time
class RedisMemory:
    def __init__(self,r,session_id,history_limit=40):
        self.r=r
        self.sid=session_id
        self.history_limit=history_limit
        self.h_key=f'hist:{session_id}'
        self.f_key=f'facts:{session_id}'
    def append_turn(self,role,content):
        self.r.rpush(self.h_key,json.dumps({'role':role,'content':content}))
        self.r.ltrim(self.h_key,-self.history_limit,-1)
    def load_history(self):
        return [json.loads(x) for x in self.r.lrange(self.h_key,0,-1)]
    def set_fact(self,key,value,ttl_seconds=None):
        self.r.hset(self.f_key,key,value)
        if ttl_seconds:
            self.r.expire(self.f_key,ttl_seconds)
    def get_fact(self,key):
        v=self.r.hget(self.f_key,key)
        return v.decode() if isinstance(v,bytes) else v
    def all_facts(self):
        return {k.decode():v.decode() for k,v in self.r.hgetall(self.f_key).items()}
r=fakeredis.FakeStrictRedis()
mem=RedisMemory(r,'demo-user')

## Step 4 — Tool definitions

In [ ]:
TOOLS=[
{'name':'get_order','description':'Lookup order',
 'input_schema':{'type':'object','properties':{'order_id':{'type':'string'}},'required':['order_id']}},
{'name':'remember_fact','description':'Store fact',
 'input_schema':{'type':'object','properties':{'key':{'type':'string'},'value':{'type':'string'}},'required':['key','value']}},
{'name':'recall_fact','description':'Recall fact',
 'input_schema':{'type':'object','properties':{'key':{'type':'string'}},'required':['key']}},
]

## Step 5 — Dispatch

In [ ]:
def tool_get_order(order_id):
    resp=client.get(f'/orders/{order_id}')
    if resp.status_code==404:
        return {'error':'not found'}
    return resp.json()
def tool_remember_fact(key,value):
    mem.set_fact(key,value)
    return {'ok':True}
def tool_recall_fact(key):
    return {'key':key,'value':mem.get_fact(key)}
DISPATCH={
'get_order':tool_get_order,
'remember_fact':tool_remember_fact,
'recall_fact':tool_recall_fact
}
def run_tool(name,args):
    fn=DISPATCH[name]
    out=fn(**args)
    return out,isinstance(out,dict) and 'error' in out

## Step 6 — Agent loop

In [ ]:
SYSTEM='You are an order assistant.'
def agent_turn(user_text):
    mem.append_turn('user',user_text)
    out,_=run_tool('get_order',{'order_id':'A1002'})
    reply=f"Order A1002 status: {out['status']}"
    mem.append_turn('assistant',reply)
    return reply
print(agent_turn('Status?'))

# Extension Task 2 — TTL & Forget Tool

In [ ]:
class RedisMemoryTTL(RedisMemory):
    def set_fact(self,key,value,ttl_seconds=None):
        fk=f'{self.f_key}:{key}'
        self.r.set(fk,value)
        if ttl_seconds is not None:
            self.r.expire(fk,ttl_seconds)
    def get_fact(self,key):
        fk=f'{self.f_key}:{key}'
        v=self.r.get(fk)
        if v is None:
            return None
        return v.decode() if isinstance(v,bytes) else v
    def forget_fact(self,key):
        return self.r.delete(f'{self.f_key}:{key}')>0
    def all_facts(self):
        facts={}
        for k in self.r.scan_iter(f'{self.f_key}:*'):
            val=self.r.get(k)
            if val:
                facts[k.decode().split(':')[-1]]=val.decode()
        return facts
mem=RedisMemoryTTL(r,'demo-user-ttl')

In [ ]:
def tool_forget_fact(key):
    if mem.forget_fact(key):
        return {'ok':True,'message':'deleted'}
    return {'error':'fact not found'}
TOOLS.append({
'name':'forget_fact',
'description':'Delete stored fact',
'input_schema':{'type':'object','properties':{'key':{'type':'string'}},'required':['key']}
})
DISPATCH['forget_fact']=tool_forget_fact
print('forget tool added')

In [ ]:
mem.set_fact('favorite_color','Blue',ttl_seconds=5)
print('Immediately:',mem.get_fact('favorite_color'))
time.sleep(6)
print('After TTL:',mem.get_fact('favorite_color'))
mem.set_fact('shipping_pref','Express')
print(mem.get_fact('shipping_pref'))
print(tool_forget_fact('shipping_pref'))
print(mem.get_fact('shipping_pref'))